# High-Throughput Sequencing Data Processing for RNA Polymerase Ribozyme Functional Validation

This notebook processes Illumina paired-end sequencing data from the **high-throughput functional assay** used to validate gRNAde-designed RNA polymerase ribozyme variants. 

The high-throughput functional assay evaluates catalytic activity of ribozyme variants by:
1. **In vitro selection**: Ribozymes ligate to templates in a functional assay
2. **Enrichment**: Active variants are enriched through primer extension and PCR amplification
3. **Deep sequencing**: Illumina sequencing quantifies variant abundance pre- and post-selection
4. **Fitness calculation**: Enrichment ratios yield fitness scores for each design

## Library Composition

**Design variants tested** (~2,000 total):
- **gRNAde designs**: Generated using gRNAde structure-conditioned language model + RibonanzaNet filtering
- **Rational designs with filtering**: Base-pairing heuristics + RibonanzaNet filtering
- **Rational designs unfiltered**: Base-pairing heuristics only

**Experimental conditions**:
- **Templates**: AUA (triplet repeats) and GAA (triplet repeats)
- **Incubation times**: Overnight (AUA), 3h and 6h (GAA)
- **Linker variants**: Short/long linkers tethering ribozyme to template
- **Pre-selection libraries**: Sequenced to normalize post-selection enrichment

## Data Structure

**Raw sequencing files** (paired-end Illumina):
- **R1/R2 FASTQ files**: Forward and reverse reads from sequencing machine
- **10 conditions**: 4 pre-selection libraries + 6 post-selection conditions (AUA overnight, GAA 3h/6h × short/long linker)

**Processing outputs**:
- **Merged reads**: Overlapping R1/R2 paired into single consensus sequences
- **Demultiplexed reads**: Separated by barcode into individual condition files
- **Filtered sequences**: Full-length variants (192 bp) with correct primer sites, reverse-complemented

## Workflow Overview

1. **Read merging & QC** (Fastp): Merge R1/R2, trim adapters, filter low-quality reads
2. **Demultiplexing** (Cutadapt): Separate reads by sample barcodes
3. **Sequence extraction**: Identify full-length sequences between primer landing sites
4. **Normalization**: Pre-selection counts normalize post-selection abundances to calculate fitness

## Step 1: Read Merging and Quality Control (Fastp)

**Fastp** merges overlapping paired-end reads (R1 + R2) into single consensus sequences and performs quality filtering.

Version: fastp 0.26.0 on MacOS

### Parameters:
- `--merge`: Combine R1 and R2 reads with sufficient overlap
- `--trim_front1/2 3` and `--trim_tail1/2 3`: Remove 3 bp from read ends to eliminate adapter contamination
- `--qualified_quality_phred 30`: Minimum Q30 base quality (99.9% accuracy)
- `--unqualified_percent_limit 5`: Reject reads with >5% low-quality bases
- `--length_required 200`: Minimum read length filter (ensures full-length ribozyme sequences)

In [ ]:
!fastp --in1 hello_S1_L001_R1_001.fastq.gz --out1 hello_S1_L001_R1_001_filt.fastq.gz \
    --in2 hello_S1_L001_R2_001.fastq.gz --out2 hello_S1_L001_R2_001_filt.fastq.gz \
    --merge --merged_out hello_S1_L001_merged.fastq.gz \
    --trim_front1 3 \
    --trim_tail1 3 \
    --trim_front2 3 \
    --trim_tail2 3 \
    --qualified_quality_phred 30 --unqualified_percent_limit 5 \
    --length_required 200

Read1 before filtering:
total reads: 11974059
total bases: 2991648702
Q20 bases: 2786499496(93.1426%)
Q30 bases: 2692090050(89.9868%)

Read2 before filtering:
total reads: 11974059
total bases: 2991193682
Q20 bases: 2477638363(82.8311%)
Q30 bases: 2224684794(74.3745%)

Merged and filtered:
total reads: 6816704
total bases: 1629762949
Q20 bases: 1626886594(99.8235%)
Q30 bases: 1614259409(99.0487%)

Filtering result:
reads passed filter: 13640674
reads failed due to low quality: 4972260
reads failed due to too many N: 0
reads failed due to too short: 5335184
reads with adapter trimmed: 17107002
bases trimmed due to adapters: 1115909971
reads corrected by overlap analysis: 1935296
bases corrected by overlap analysis: 4822706

Duplication rate: 4.13688%

Insert size peak (evaluated by paired-end reads): 209

Read pairs merged: 6816704
% of original read pairs: 56.9289%
% in reads after filtering: 100%


JSON report: fastp.json
HTML report: fastp.html

fastp --in1 hello_S1_L001_R1_001.fastq

## Step 2: Demultiplexing with Cutadapt

**Cutadapt** separates pooled sequencing reads into individual samples based on unique barcode sequences.

Version: cutadapt 5.1 on MacOS

### Demultiplexing Strategy:
Each of the 10 experimental conditions (4 pre-selection + 6 post-selection) is identified by a unique 5' barcode sequence embedded in the amplicon. Cutadapt searches for these barcodes and routes reads to condition-specific output files.

### Barcode File Format:
The `barcodes.fasta` file contains barcode sequences for each condition:
```
>pre_AUA_shortL
ATCACGGATGCCATGCCGACCC
>pre_AUA_longL
CGATGTGATGCCATGCCGACCC
>pre_GAA_shortL
TTAGGCGATGCCATGCCGACCC
>pre_GAA_longL
TGACCAGATGCCATGCCGACCC
>post_AUA_on_shortL
ACAGTGGATGCCATGCCGACCC
>post_AUA_on_longL
GCCAATGATGCCATGCCGACCC
>post_GAA_3h_shortL
CAGATCGATGCCATGCCGACCC
>post_GAA_3h_longL
ACTTGAGATGCCATGCCGACCC
>post_GAA_6h_shortL
GATCAGGATGCCATGCCGACCC
>post_GAA_6h_longL
TAGCTTGATGCCATGCCGACCC
```

### Cutadapt Parameters:
- `-g file:barcodes.fasta`: Search for 5' barcodes from file
- `--no-indels`: Require exact barcode matches (no insertions/deletions)
- `-e 0`: Zero error tolerance for barcode matching (perfect matches only)
- `--action=retain`: Keep barcode in output sequence for downstream validation
- `-o demultiplexed_reads_merged/{name}.fasta`: Write to sample-specific files

In [6]:
!mkdir -p demultiplexed_reads_merged

In [7]:
!cutadapt -g file:barcodes.fasta --no-indels -e 0 --action=retain -o demultiplexed_reads_merged/{name}.fasta hello_S1_L001_merged.fastq.gz > demultiplex_report.txt

Done           00:01:52     6,816,704 reads @  16.5 µs/read;   3.64 M reads/minute


## Step 3: Sequence Processing Utilities

This section defines utility functions for DNA/RNA sequence manipulation required for downstream processing.

### Function Descriptions

**Core sequence manipulation:**
- `rc_dna(sequence)`: Reverse complement for standard DNA bases (A↔T, G↔C)
- `rc_dna_iupac(sequence)`: Reverse complement supporting IUPAC ambiguity codes (e.g., R, Y, K, M)
- `rna2dna(sequence)`: Convert RNA to DNA (U→T)
- `dna2rna(sequence)`: Convert DNA to RNA (T→U)

**FASTA file I/O:**
- `readFasta(fastaFile)`: Generator function to parse FASTA files line-by-line (memory-efficient for large files)

**Oligonucleotide design utilities:**
- `spike_oligo(percentage, sequence)`: Generate hand-mix strings for oligo synthesis with specified mutation rates
- `handmixN(sequence)`: Replace N with equal 25% mix of A/G/C/T
- `handmixNbcg(sequence)`: Replace N with biased mix (20% A, 30% G/C, 20% T) to reduce homopolymer runs

In [8]:
import sys
import os
import pandas as pd
import re
from collections import Counter
from scipy.stats.mstats import gmean
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

def rc_dna(sequence):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    seq = sequence
    reverse_complement = "".join(complement.get(base, base) for base in reversed(seq))
    return(reverse_complement)

def rc_dna_iupac(sequence):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A','R':'Y','Y':'R','K':'M', 'M':'K', 'B':'V','V':'B','D':'H','H':'D','S':'S','W':'W'}
    seq = sequence
    reverse_complement = "".join(complement.get(base, base) for base in reversed(seq))
    return(reverse_complement)

def spike_oligo(percentage,sequence):
    DNA_dict={'A':0,'C':0,'G':0,'T':0}
    spiked_str =''
    for i, nt in enumerate(sequence):
        DNA_nt = ['A','C','G','T']
        DNA_dict[nt] = 100-percentage
        DNA_nt.remove(nt)
    
        for i in DNA_nt:
            DNA_dict[i]=int(percentage/3)
        
        DNA_nt = ['A', 'C','G','T']
        temp_list = ('(',''.join([str(DNA_dict[x]).zfill(2) for x in DNA_nt]),')')
        spiked_str += "".join(temp_list)
    return spiked_str

# def to read fasta files
def readFasta(fastaFile):
    fh = open(fastaFile, 'r')
    for line in fh:
        if line[0] == '>':
            header = line.rstrip()[1:]
            if sys.version_info[0] < 3:
                seq = fh.next().rstrip()
            else:
                seq = fh.readline().rstrip()
        yield [header, seq]
    fh.close()
    
def rna2dna(sequence):
    invtx = {'U':'T'}
    seq = sequence
    reverse_complement = "".join(invtx.get(base, base) for base in (seq))
    return(reverse_complement)

def dna2rna(sequence):
    tx = {'T':'U'}
    seq = sequence
    reverse_complement = "".join(tx.get(base, base) for base in (seq))
    return(reverse_complement)

def handmixN(sequence):
    invtx = {'N':'(25252525)'}
    seq = sequence
    reverse_complement = "".join(invtx.get(base, base) for base in (seq))
    return(reverse_complement)

def handmixNbcg(sequence):
    invtx = {'N':'(20303020)'}
    seq = sequence
    reverse_complement = "".join(invtx.get(base, base) for base in (seq))
    return(reverse_complement)
    

## Step 4: Full-Length Sequence Extraction and Filtering

This step extracts only **full-length, correctly-amplified ribozyme sequences** from demultiplexed reads.

### Primer Landing Sites
The functional assay amplicon contains invariant primer binding sites flanking the variable ribozyme sequence:
- **Left primer**: `GATGCCATGCCGACCC` (5' end)
- **Right primer**: `CCTGTTTGTTTGTTTTGTTGTTTGTT` (3' end)

### Filtering Criteria
Only sequences meeting ALL criteria are retained:
1. **Both primers present**: Ensures complete amplification (no truncated products)
2. **Exact primer match**: Requires perfect sequence identity at landing sites (validates PCR specificity)
3. **Correct length** (192 bp): Full-length ribozyme + primers (rejects deletion/insertion artifacts)

### Processing Steps
For each demultiplexed FASTA file:
1. **Locate primers**: Find exact matches to both primer sequences
2. **Extract amplicon**: Substring between primers (inclusive)
3. **Reverse complement**: Convert to sense strand orientation (sequencing yields antisense)
4. **Length check**: Verify 192 bp total length
5. **Output**: Save to `rctrim/` folder with statistics

### Experimental Conditions Processed
```
Pre-selection libraries (4):        Post-selection samples (6):
- pre_AUA_shortL                    - post_AUA_on_shortL
- pre_AUA_longL                     - post_AUA_on_longL  
- pre_GAA_shortL                    - post_GAA_3h_shortL
- pre_GAA_longL                     - post_GAA_3h_longL
                                    - post_GAA_6h_shortL
                                    - post_GAA_6h_longL
```

In [ ]:
from tqdm.auto import tqdm

# reverse complement and trim full length samples
left = 'GATGCCATGCCGACCC'
right   = 'CCTGTTTGTTTGTTTTGTTGTTTGTT' 
both = [left ,right]

# folder where the further processed reads go (trimmed to full-length and reverse complemented)
os.makedirs(os.path.join(os.getcwd(), "demultiplexed_reads_merged","rctrim"), exist_ok=True) 

rounds_names =  [
    "pre_AUA_shortL",
    "pre_AUA_longL",
    "pre_GAA_shortL",
    "pre_GAA_longL",
    "post_AUA_on_shortL",
    "post_AUA_on_longL",
    "post_GAA_3h_shortL",
    "post_GAA_3h_longL",
    "post_GAA_6h_shortL",
    "post_GAA_6h_longL"
]

for n,i in tqdm(enumerate(rounds_names)):
    count = 0
    count_tot = 0
    handle_fasta = open(('./demultiplexed_reads_merged/rctrim/'+rounds_names[n]+'_rc_trim_pfl.fasta'), 'w')
    for line in readFasta('./demultiplexed_reads_merged/'+rounds_names[n]+'.fasta'):
        count_tot+=1
        seq = line[1]
        if all(x in seq for x in both): # ensures that only sequences with the perfect primer landing sites are present
            left_index = seq.index(left)
            right_index = seq.index(right)
            tmp_seq = (seq[left_index:right_index+len(right)])
            rc_tmp_seq = rc_dna(tmp_seq)
            if len(tmp_seq)==192: 
                count+=1
                handle_fasta.write(">%s\n" % (line[0]))
                handle_fasta.write("%s\n" % rc_tmp_seq)
    print (i,count_tot, count, count/count_tot)
    handle_fasta.close()

0it [00:00, ?it/s]

pre_AUA_shortL 497961 399360 0.8019905173296704
pre_AUA_longL 524323 415080 0.7916494222073035
pre_GAA_shortL 557944 453830 0.8133970434308819
pre_GAA_longL 603898 493718 0.8175519706970382
post_AUA_on_shortL 451823 404610 0.8955055408865861
post_AUA_on_longL 592556 530326 0.8949803900390849
post_GAA_3h_shortL 569510 508533 0.892930765043634
post_GAA_3h_longL 430770 386138 0.8963901850175268
post_GAA_6h_shortL 464444 414174 0.8917630543187123
post_GAA_6h_longL 443676 395173 0.8906792343962712
